# AI-Generated Essay Detection

A series of four interpretable tests that together produce a combined `ai_risk_score` for each
real admissions essay. High-scoring essays are flagged for qualitative review.

**Tests:**
1. Cosine similarity to synthetic embedding space
2. GPT-2 perplexity + burstiness
3. Stylometric flatness
4. Discourse patterns
5. Combined risk score + flagging

**Run order:** Sections 3 and 4 use pre-computed features and are fast. Section 1 requires
GPU encoding (~15 min). Section 2 requires GPT-2 inference (~1–2 hours) and uses checkpointing.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT    = Path('..')
OUTPUTS = Path('../outputs')

REAL_STYLO  = OUTPUTS / 'stylometrics/stylometric_features.csv'
REAL_DISC   = OUTPUTS / 'discourse/real/full_features.csv'
SYN_DISC    = OUTPUTS / 'discourse/synthetic/full_features.csv'
REAL_TOPIC  = OUTPUTS / 'topics/real/essay_topics.csv'

SYN_ESSAYS  = Path('../data/synthetic/synthetic_essays.csv')
FF_ESSAYS   = Path('../data/freeform/freeform_essays.csv')
ESSAYS_PATH = ROOT / 'data/real/essays.xlsx'   # loaded for text only, never written out

OUTPUT_DIR  = OUTPUTS / 'ai_detection'
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'test_summaries').mkdir(exist_ok=True)

# ── Config ────────────────────────────────────────────────────────────────────
EMBED_MODEL      = 'all-MiniLM-L6-v2'   # sentence-transformers model for Test 1
TOP_K_SIM        = 10                    # top-K neighbors for cosine similarity
PERP_CHECKPOINT  = OUTPUT_DIR / 'perplexity_checkpoint.csv'

# ── Helpers ───────────────────────────────────────────────────────────────────
_PREFIX_MAP = {"rea": "real", "syn": "synthetic", "fre": "freeform"}

def corpus_from_id(student_id) -> str:
    return _PREFIX_MAP.get(str(student_id)[:3], "unknown")

In [ ]:
# ── Build master dataframe (real essays, pre-computed features) ───────────────
stylo = pd.read_csv(REAL_STYLO)
stylo['corpus'] = stylo['student_id'].map(corpus_from_id)

real_stylo = stylo[stylo['corpus'] == 'real'].copy()
syn_stylo  = stylo[stylo['corpus'] == 'synthetic'].copy()
ff_stylo   = stylo[stylo['corpus'] == 'freeform'].copy()

# ── Build row_idx → rea_CAMPUSID mapping ─────────────────────────────────────
# Discourse/topic CSVs use the original DataFrame row index, not student ID.
# Essays file rows with essays present map to the stylometrics student_id format.
essays_idx = pd.read_excel(ESSAYS_PATH, usecols=['Campus ID', 'Essay Response'])
essays_idx = essays_idx[essays_idx['Essay Response'].notna()].reset_index()
essays_idx.rename(columns={'index': 'row_idx', 'Campus ID': 'campus_id'}, inplace=True)
essays_idx['student_id'] = 'rea_' + essays_idx['campus_id'].astype(str)
row_to_sid = essays_idx.set_index('row_idx')['student_id'].to_dict()

# ── Merge discourse features ──────────────────────────────────────────────────
disc_real = pd.read_csv(REAL_DISC)
disc_real['student_id'] = disc_real['student_id'].map(row_to_sid)

master = real_stylo.merge(disc_real[['student_id', 'discourse_cluster', 'arc_label',
                                      'n_claims', 'n_evidence', 'arg_density',
                                      'claim_evidence_ratio', 'coherence_score']],
                           on='student_id', how='left')

# ── Merge topic features ──────────────────────────────────────────────────────
if REAL_TOPIC.exists():
    topics = pd.read_csv(REAL_TOPIC)[['student_id', 'topic']]
    topics['student_id'] = topics['student_id'].map(row_to_sid)
    master = master.merge(topics, on='student_id', how='left')

print(f'Master dataframe: {len(master):,} real essays, {master.shape[1]} columns')
print(f'  Discourse features merged: {"coherence_score" in master.columns}')
print(f'  Topic feature merged:      {"topic" in master.columns}')
print(f'  Discourse NaN rate: {master["coherence_score"].isna().mean():.1%}')
print(master.head(2))

## Test 1: Cosine Similarity to Synthetic Embedding Space

**Signal:** Does this essay have a near-duplicate match in our synthetic corpus?  
**Method:** Encode all synthetic + freeform essays as a reference pool. For each real essay, compute its maximum cosine similarity to any individual essay in the reference pool (nearest-neighbor). Flag essays that exceed `NN_THRESHOLD` (default 0.85, calibrated against the within-synthetic similarity distribution). A flagged essay is paired with its closest synthetic neighbor for auditing.  
**Model:** `all-MiniLM-L6-v2` (already cached from discourse coherence section)

> **Note:** An earlier approach (`t1`) used mean similarity to the top-10 nearest neighbors. The pairwise nearest-neighbor approach (`t1b`) is preferred because it is more sensitive to near-duplicate matches and produces an auditable essay pair rather than an aggregate score.


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ── Load essays (text held in memory only during this section) ────────────────
essays_df = pd.read_excel(ESSAYS_PATH, usecols=['Campus ID', 'Essay Response'])
essays_df = essays_df.rename(columns={'Campus ID': 'student_id', 'Essay Response': 'essay'})
essays_df['student_id'] = essays_df['student_id'].astype(str)

syn_essays = pd.read_csv(SYN_ESSAYS)[['student_id', 'essay']]
ff_essays  = pd.read_csv(FF_ESSAYS)[['student_id', 'essay']]

# Strip corpus prefix (e.g. 'rea_6290255' -> '6290255') for joining against essays.xlsx
master_join = master[['student_id']].copy()
master_join['student_id_raw'] = master_join['student_id'].str.replace(r'^[a-z]+_', '', regex=True)

real_texts = (
    master_join
    .merge(essays_df.rename(columns={'student_id': 'student_id_raw'}), on='student_id_raw', how='left')
    [['student_id', 'essay']]
)

# Reference pool: all synthetic + freeform essays
ref_essays = pd.concat([syn_essays, ff_essays], ignore_index=True)

print(f'Real essays to encode:      {len(real_texts):,}')
print(f'Essays with text matched:   {real_texts["essay"].notna().sum():,}')
print(f'Reference essays (AI pool): {len(ref_essays):,}')

# ── Encode ────────────────────────────────────────────────────────────────────
model = SentenceTransformer(EMBED_MODEL)

print('Encoding reference (AI) pool...')
ref_embs = model.encode(ref_essays['essay'].tolist(), batch_size=64,
                         show_progress_bar=True, normalize_embeddings=True)

print('Encoding real essays...')
real_embs = model.encode(real_texts['essay'].fillna('').tolist(), batch_size=64,
                          show_progress_bar=True, normalize_embeddings=True)

# ── Compute similarity scores ─────────────────────────────────────────────────
sim_matrix = real_embs @ ref_embs.T

top_k_sim = np.sort(sim_matrix, axis=1)[:, -TOP_K_SIM:].mean(axis=1)
centroid   = ref_embs.mean(axis=0, keepdims=True)
centroid  /= np.linalg.norm(centroid)
centroid_sim = (real_embs @ centroid.T).flatten()

master['t1_cos_sim_top10'] = top_k_sim
master['t1_cos_sim_centroid'] = centroid_sim

# Calibrate: 95th percentile of synthetic-to-synthetic within-corpus similarity
syn_self_sim = np.sort(ref_embs @ ref_embs.T, axis=1)[:, -TOP_K_SIM-1:-1].mean(axis=1)
t1_threshold = np.percentile(syn_self_sim, 95)
print(f'\nTest 1 calibration threshold (95th pct within-synthetic sim): {t1_threshold:.4f}')

master['t1_score'] = np.clip(
    (top_k_sim - top_k_sim.min()) / (top_k_sim.max() - top_k_sim.min()), 0, 1
)

print(f'Real essays mean top-10 sim: {top_k_sim.mean():.4f} +/- {top_k_sim.std():.4f}')
print(f'Essays above calibration threshold: {(top_k_sim > t1_threshold).sum():,}')

t1_summary = pd.DataFrame([{
    'n_real': len(master),
    'n_ref_pool': len(ref_essays),
    'embed_model': EMBED_MODEL,
    'top_k': TOP_K_SIM,
    'threshold_95pct': t1_threshold,
    'mean_sim': top_k_sim.mean(),
    'std_sim': top_k_sim.std(),
    'p50': np.percentile(top_k_sim, 50),
    'p90': np.percentile(top_k_sim, 90),
    'p95': np.percentile(top_k_sim, 95),
    'n_flagged': int((top_k_sim > t1_threshold).sum()),
}])
t1_summary.to_csv(OUTPUT_DIR / 'test_summaries/t1_cosine_summary.csv', index=False)

del real_embs, ref_embs, sim_matrix, syn_self_sim, essays_df

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(top_k_sim, bins=60, color='#4C72B0', alpha=0.8)
ax.axvline(t1_threshold, color='red', linestyle='--', label=f'Threshold ({t1_threshold:.3f})')
ax.set_xlabel(f'Mean cosine similarity to top-{TOP_K_SIM} synthetic neighbors')
ax.set_ylabel('Number of essays')
ax.set_title('Test 1: Cosine similarity distribution (real essays)')
ax.legend()
plt.tight_layout()
plt.show()

### Test 1b: Nearest-Neighbor Pair Detection

For each real essay, find its single closest synthetic neighbor and flag if the similarity exceeds `NN_THRESHOLD`. The flagged output includes the matching synthetic `student_id` so reviewers can inspect the pair directly.


In [ ]:
# ── Test 1b: Nearest-neighbor exact match detection ──────────────────────────
NN_THRESHOLD = 0.85   # tune after seeing the distribution below

model_nn = SentenceTransformer(EMBED_MODEL)

ref_essays_nn = pd.concat([pd.read_csv(SYN_ESSAYS)[['student_id', 'essay']],
                            pd.read_csv(FF_ESSAYS)[['student_id', 'essay']]], ignore_index=True)

master_join_nn = master[['student_id']].copy()
master_join_nn['student_id_raw'] = master_join_nn['student_id'].str.replace(r'^[a-z]+_', '', regex=True)
essays_df_nn = pd.read_excel(ESSAYS_PATH, usecols=['Campus ID', 'Essay Response'])
essays_df_nn = essays_df_nn.rename(columns={'Campus ID': 'student_id', 'Essay Response': 'essay'})
essays_df_nn['student_id'] = essays_df_nn['student_id'].astype(str)
real_texts_nn = (
    master_join_nn
    .merge(essays_df_nn.rename(columns={'student_id': 'student_id_raw'}), on='student_id_raw', how='left')
    [['student_id', 'essay']]
).reset_index(drop=True)
del essays_df_nn

print('Encoding reference pool...')
ref_embs_nn = model_nn.encode(ref_essays_nn['essay'].tolist(), batch_size=64,
                               show_progress_bar=True, normalize_embeddings=True)
print('Encoding real essays...')
real_embs_nn = model_nn.encode(real_texts_nn['essay'].fillna('').tolist(), batch_size=64,
                                show_progress_bar=True, normalize_embeddings=True)

sim_matrix_nn = np.array(real_embs_nn @ ref_embs_nn.T)
nn_sim = sim_matrix_nn.max(axis=1).flatten()
nn_idx = sim_matrix_nn.argmax(axis=1).flatten()

master['t1b_nn_sim']  = nn_sim
master['t1b_flagged'] = nn_sim >= NN_THRESHOLD

# ── Distribution — use this to pick a threshold ───────────────────────────────
print('\nNN similarity distribution (real essays vs best synthetic match):')
for pct in [50, 75, 90, 95, 99, 99.5, 100]:
    print(f'  p{pct:5.1f}: {np.percentile(nn_sim, pct):.4f}')

n_flagged = int(master['t1b_flagged'].sum())
print(f'\nFlagged at {NN_THRESHOLD}: {n_flagged:,} / {len(master):,} ({n_flagged/len(master):.1%})')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(nn_sim, bins=80, color='#4C72B0', alpha=0.8)
ax.axvline(NN_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({NN_THRESHOLD})')
ax.set_xlabel('Max cosine similarity to any synthetic essay (nearest neighbor)')
ax.set_ylabel('Number of essays')
ax.set_title('Test 1b: Nearest-neighbor similarity distribution')
ax.legend()
plt.tight_layout()
plt.show()

# ── Flagged pairs ─────────────────────────────────────────────────────────────
flagged_idx = np.where(nn_sim >= NN_THRESHOLD)[0]
pairs = []
for i in flagged_idx:
    ref_i = int(nn_idx[i])
    pairs.append({
        'student_id':  real_texts_nn.iloc[i]['student_id'],
        'nn_sim':      float(nn_sim[i]),
        'matched_ref': ref_essays_nn.iloc[ref_i]['student_id'],
        'real_excerpt': str(real_texts_nn.iloc[i]['essay'])[:200],
        'ref_excerpt':  str(ref_essays_nn.iloc[ref_i]['essay'])[:200],
    })

if pairs:
    pairs_df = pd.DataFrame(pairs).sort_values('nn_sim', ascending=False)
    pairs_df.to_csv(OUTPUT_DIR / 'test_summaries/t1b_nn_flagged_pairs.csv', index=False)
    display(pairs_df[['student_id', 'nn_sim', 'matched_ref', 'real_excerpt']].head(20))
else:
    print(f'No essays flagged at threshold {NN_THRESHOLD} — try lowering NN_THRESHOLD based on distribution above.')

del real_embs_nn, ref_embs_nn, sim_matrix_nn

## Test 2: GPT-2 Perplexity + Burstiness

**Signal:** LLMs produce uniformly low-perplexity text; humans are "bursty."  
**Method:** GPT-2 overall perplexity (lower = more LLM-like) + coefficient of variation
of per-sentence perplexity (lower = more uniform = more LLM-like).  
**Runtime:** ~1–2 hours on GPU for 15k essays. Uses checkpointing — safe to interrupt and resume.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import spacy

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

gpt2_model = AutoModelForCausalLM.from_pretrained('gpt2').to(device).eval()
gpt2_tok   = AutoTokenizer.from_pretrained('gpt2')
gpt2_tok.pad_token = gpt2_tok.eos_token

nlp = spacy.load('en_core_web_sm', disable=['ner', 'lemmatizer'])


def gpt2_perplexity(text: str, max_tokens: int = 1024, stride: int = 512) -> float:
    """Sliding-window GPT-2 perplexity (handles essays longer than 1024 tokens)."""
    enc = gpt2_tok(text, return_tensors='pt', truncation=False)
    input_ids = enc['input_ids'][0]
    n = len(input_ids)
    if n == 0:
        return float('nan')

    total_nll, total_tokens = 0.0, 0
    for begin in range(0, n, stride):
        end   = min(begin + max_tokens, n)
        chunk = input_ids[begin:end].unsqueeze(0).to(device)
        with torch.no_grad():
            loss = gpt2_model(chunk, labels=chunk).loss
        chunk_len = end - begin
        total_nll    += loss.item() * chunk_len
        total_tokens += chunk_len
        if end == n:
            break
    return float(np.exp(total_nll / total_tokens))


def perplexity_burstiness(text: str) -> tuple[float, float]:
    """Returns (essay_perplexity, burstiness_cv) where cv = std/mean of sentence perplexities."""
    doc  = nlp(text)
    sents = [s.text.strip() for s in doc.sents if len(s.text.strip()) > 10]
    if len(sents) < 3:
        return gpt2_perplexity(text), float('nan')
    sent_perps = [gpt2_perplexity(s) for s in sents]
    sent_perps = [p for p in sent_perps if not np.isnan(p) and np.isfinite(p)]
    if not sent_perps:
        return float('nan'), float('nan')
    essay_perp = gpt2_perplexity(text)
    cv = np.std(sent_perps) / np.mean(sent_perps) if np.mean(sent_perps) > 0 else float('nan')
    return essay_perp, cv

print('GPT-2 and spaCy loaded.')

In [ ]:
# ── Load real essay texts for perplexity computation ─────────────────────────
essays_df = pd.read_excel(ESSAYS_PATH, usecols=['Campus ID', 'Essay Response'])
essays_df = essays_df.rename(columns={'Campus ID': 'student_id', 'Essay Response': 'essay'})
essays_df['student_id'] = essays_df['student_id'].astype(str)

master_join = master[['student_id']].copy()
master_join['student_id_raw'] = master_join['student_id'].str.replace(r'^[a-z]+_', '', regex=True)
real_texts = (
    master_join
    .merge(essays_df.rename(columns={'student_id': 'student_id_raw'}), on='student_id_raw', how='left')
    [['student_id', 'essay']]
)
del essays_df

# ── Resume from checkpoint if it exists ───────────────────────────────────────
if PERP_CHECKPOINT.exists():
    done = pd.read_csv(PERP_CHECKPOINT)
    done_ids = set(done['student_id'])
    print(f'Resuming from checkpoint: {len(done_ids):,} essays already processed.')
else:
    done = pd.DataFrame(columns=['student_id', 'essay_perplexity', 'burstiness_cv'])
    done_ids = set()

remaining = real_texts[~real_texts['student_id'].isin(done_ids)]
print(f'Remaining: {len(remaining):,} essays to process.')

# ── Run inference (with checkpointing every 100 essays) ───────────────────────
results = []
for i, row in enumerate(remaining.itertuples()):
    text = str(row.essay) if pd.notna(row.essay) else ''
    perp, bursty = perplexity_burstiness(text)
    results.append({'student_id': row.student_id,
                    'essay_perplexity': perp,
                    'burstiness_cv': bursty})
    if (i + 1) % 100 == 0:
        batch_df = pd.DataFrame(results)
        updated = pd.concat([done, batch_df], ignore_index=True)
        updated.to_csv(PERP_CHECKPOINT, index=False)
        print(f'  Checkpoint saved: {len(done) + len(results):,}/{len(real_texts):,}')

# Save final checkpoint
perp_df = pd.concat([done, pd.DataFrame(results)], ignore_index=True)
perp_df.to_csv(PERP_CHECKPOINT, index=False)
print(f'Perplexity computation complete: {len(perp_df):,} essays.')
del real_texts

In [ ]:
# ── Calibrate thresholds using synthetic corpus ────────────────────────────────
syn_essays = pd.read_csv(SYN_ESSAYS)[['student_id', 'essay']]
ff_essays  = pd.read_csv(FF_ESSAYS)[['student_id', 'essay']]
ref_essays = pd.concat([syn_essays, ff_essays], ignore_index=True)

print('Computing perplexity on synthetic reference pool for calibration...')
ref_perp_cache = OUTPUT_DIR / 'ref_perplexity_cache.csv'
if ref_perp_cache.exists():
    ref_perp_df = pd.read_csv(ref_perp_cache)
    print(f'  Loaded cached reference perplexity ({len(ref_perp_df):,} essays).')
else:
    ref_rows = []
    for i, row in enumerate(ref_essays.itertuples()):
        text = str(row.essay) if pd.notna(row.essay) else ''
        perp, bursty = perplexity_burstiness(text)
        ref_rows.append({'student_id': row.student_id, 'essay_perplexity': perp, 'burstiness_cv': bursty})
        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(ref_essays)}')
    ref_perp_df = pd.DataFrame(ref_rows)
    ref_perp_df.to_csv(ref_perp_cache, index=False)

# Thresholds: 10th percentile of synthetic distribution (essays below this are LLM-like)
t2_perp_threshold   = np.nanpercentile(ref_perp_df['essay_perplexity'], 10)
t2_bursty_threshold = np.nanpercentile(ref_perp_df['burstiness_cv'], 10)
print(f'Test 2 thresholds — perplexity: {t2_perp_threshold:.1f}  |  burstiness CV: {t2_bursty_threshold:.3f}')

# ── Merge into master and compute score ──────────────────────────────────────
perp_df = pd.read_csv(PERP_CHECKPOINT)
master = master.merge(perp_df[['student_id', 'essay_perplexity', 'burstiness_cv']],
                       on='student_id', how='left')

# Perplexity score: normalized rank (low perplexity = high risk)
perp_rank = master['essay_perplexity'].rank(pct=True)
bursty_rank = master['burstiness_cv'].rank(pct=True)
master['t2_perplexity_flag'] = (
    (master['essay_perplexity'] < t2_perp_threshold) &
    (master['burstiness_cv'] < t2_bursty_threshold)
).astype(int)
# Combined score: low perplexity rank + low burstiness rank = high risk
master['t2_score'] = np.clip(1 - (perp_rank + bursty_rank) / 2, 0, 1)

print(f'\nFlagged by Test 2: {master["t2_perplexity_flag"].sum():,} essays')

pd.DataFrame([{
    'perp_threshold_p10': t2_perp_threshold,
    'bursty_threshold_p10': t2_bursty_threshold,
    'n_flagged': int(master['t2_perplexity_flag'].sum()),
    'mean_real_perp': master['essay_perplexity'].mean(),
    'mean_syn_perp': ref_perp_df['essay_perplexity'].mean(),
}]).to_csv(OUTPUT_DIR / 'test_summaries/t2_perplexity_summary.csv', index=False)

del ref_essays, syn_essays, ff_essays

## Test 3: Stylometric Flatness

**Signal:** High vocabulary richness + low sentence-length variation = LLM signature.  
**Method:** Rule-based flag (count of conditions met) + continuous z-score composite.  
**Source:** Already-computed stylometric features — no new inference needed.

> Revise feature weights below after running Section 8 of `stylometrics.ipynb`.

In [ ]:
# ── Load synthetic stylometric features for threshold calibration ─────────────
stylo = pd.read_csv(REAL_STYLO)
stylo['corpus'] = stylo['student_id'].map(corpus_from_id)
syn_stylo = stylo[stylo['corpus'].isin(['synthetic', 'freeform'])].copy()

# ── Feature weights from Section 8 Cohen's d (absolute values) ───────────────
T3_FEATURES = {
    'mtld':                  {'d': 1.492, 'direction': +1},  # synthetic > real
    'hdd':                   {'d': 1.185, 'direction': +1},
    'avg_word_length':        {'d': 0.906, 'direction': +1},
    'anaphora_proxy':         {'d': 0.531, 'direction': +1},
    'subordination_ratio':    {'d': 0.500, 'direction': -1},  # real > synthetic, flip
    'rhetorical_q_rate':      {'d': 0.454, 'direction': -1},
    'hollow_phrase_density':  {'d': 0.285, 'direction': +1},
    'decorative_adj_density': {'d': 0.285, 'direction': +1},
    'cv_sent_len':            {'d': 0.296, 'direction': -1},
}
available_feats = {f: v for f, v in T3_FEATURES.items()
                   if f in master.columns and f in syn_stylo.columns}

# ── Threshold calibration from synthetic distribution ─────────────────────────
t3_mtld_threshold       = np.nanpercentile(syn_stylo['mtld'], 75)
t3_hdd_threshold        = np.nanpercentile(syn_stylo['hdd'], 75) if 'hdd' in syn_stylo.columns else None
t3_rhetorical_threshold = np.nanpercentile(syn_stylo['rhetorical_q_rate'], 25)
t3_cv_threshold         = np.nanpercentile(syn_stylo['cv_sent_len'], 25) if 'cv_sent_len' in syn_stylo.columns else None
t3_hollow_threshold     = np.nanpercentile(syn_stylo['hollow_phrase_density'], 75) if 'hollow_phrase_density' in syn_stylo.columns else None

print('Test 3 thresholds (from synthetic/freeform distribution):')
print(f'  mtld > {t3_mtld_threshold:.2f}')
if t3_hdd_threshold:    print(f'  hdd > {t3_hdd_threshold:.4f}')
if t3_cv_threshold:     print(f'  cv_sent_len < {t3_cv_threshold:.4f}')
if t3_hollow_threshold: print(f'  hollow_phrase_density > {t3_hollow_threshold:.4f}')
print(f'  rhetorical_q_rate < {t3_rhetorical_threshold:.4f}')

# ── Helper: z-score using combined real+synthetic distribution ────────────────
# Use pd.api.types to avoid dtype string comparison issues in newer pandas
import pandas.api.types as pat
shared_cols = [c for c in master.columns
               if c in syn_stylo.columns and pat.is_numeric_dtype(master[c])]
all_essays_feats = pd.concat([master[shared_cols], syn_stylo[shared_cols]])
feat_means = all_essays_feats.mean()
feat_stds  = all_essays_feats.std().replace(0, 1)

def disc_zscore(series, feat):
    return (series - feat_means[feat]) / feat_stds[feat]

# ── Rule-based flag: 2+ of 3 conditions ──────────────────────────────────────
cond_mtld       = master['mtld'] > t3_mtld_threshold
cond_hdd        = (master['hdd'] > t3_hdd_threshold) if t3_hdd_threshold is not None else pd.Series(False, index=master.index)
cond_rhetorical = master['rhetorical_q_rate'] < t3_rhetorical_threshold

master['t3_conditions_met'] = cond_mtld.astype(int) + cond_hdd.astype(int) + cond_rhetorical.astype(int)
master['t3_flatness_flag']  = (master['t3_conditions_met'] >= 2).astype(int)

# ── Continuous score: Cohen's d-weighted z-score composite ───────────────────
total_weight = sum(v['d'] for v in available_feats.values())

t3_raw = pd.Series(0.0, index=master.index)
for feat, meta in available_feats.items():
    z = (master[feat] - feat_means[feat]) / feat_stds[feat]
    t3_raw += meta['d'] * meta['direction'] * z
t3_raw /= total_weight

master['t3_score'] = (t3_raw - t3_raw.min()) / (t3_raw.max() - t3_raw.min())

print(f'\nFlagged by Test 3: {master["t3_flatness_flag"].sum():,} essays (2+ of 3 conditions)')
print('\nTest 3 score statistics:')
print(master['t3_score'].describe().round(3))

# ── Scatter: mtld vs. hdd ─────────────────────────────────────────────────────
if 'hdd' in master.columns:
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(master['hdd'], master['mtld'], c=master['t3_score'],
                    cmap='YlOrRd', alpha=0.3, s=8)
    ax.axhline(t3_mtld_threshold, color='blue', linestyle='--', alpha=0.7, label='mtld threshold')
    if t3_hdd_threshold:
        ax.axvline(t3_hdd_threshold, color='green', linestyle='--', alpha=0.7, label='hdd threshold')
    ax.scatter(syn_stylo['hdd'], syn_stylo['mtld'],
               color='orange', alpha=0.5, s=15, marker='x', label='synthetic ref')
    plt.colorbar(sc, ax=ax, label='t3_score')
    ax.set_xlabel('hdd'); ax.set_ylabel('mtld')
    ax.set_title('Test 3: Top two stylometric discriminators')
    ax.legend(fontsize=8); plt.tight_layout(); plt.show()

pd.DataFrame([{
    'features_used': ', '.join(available_feats.keys()),
    'mtld_threshold': t3_mtld_threshold,
    'hdd_threshold': t3_hdd_threshold,
    'rhetorical_q_threshold': t3_rhetorical_threshold,
    'cv_sent_len_threshold': t3_cv_threshold,
    'hollow_phrase_threshold': t3_hollow_threshold,
    'n_flagged_2plus': int(master['t3_flatness_flag'].sum()),
    'n_flagged_3of3':  int((master['t3_conditions_met'] == 3).sum()),
}]).to_csv(OUTPUT_DIR / 'test_summaries/t3_stylometric_summary.csv', index=False)

## Test 4: Discourse Patterns

**Signal:** LLMs produce overly coherent, more argument-dense essays.  
**Method:** Weighted z-score composite of `coherence_score` and `arg_density`.  
**Source:** Already-computed discourse features — no new inference needed.

> Revise feature weights below after running Section 6 of `discourse_analysis.ipynb`.

In [ ]:
# ── Load synthetic discourse features for threshold calibration ───────────────
syn_disc = pd.read_csv(SYN_DISC)

disc_feats = ['coherence_score', 'arg_density', 'claim_evidence_ratio']
disc_feats = [f for f in disc_feats if f in master.columns and f in syn_disc.columns]

combined_disc = pd.concat([
    master[disc_feats],
    syn_disc[disc_feats]
])
disc_means = combined_disc.mean()
disc_stds  = combined_disc.std().replace(0, 1)

def disc_zscore(col, feat):
    return (col - disc_means[feat]) / disc_stds[feat]

# ── Composite: high coherence + high arg_density = AI-like ────────────────────
score_parts = []
if 'coherence_score' in disc_feats:
    score_parts.append(disc_zscore(master['coherence_score'], 'coherence_score'))
if 'arg_density' in disc_feats:
    score_parts.append(disc_zscore(master['arg_density'], 'arg_density'))

if score_parts:
    raw_score = sum(score_parts) / len(score_parts)
    master['t4_score'] = np.clip(
        (raw_score - raw_score.min()) / (raw_score.max() - raw_score.min()), 0, 1
    )
    t4_flag_threshold = raw_score.mean() + 1.5 * raw_score.std()
    master['t4_discourse_flag'] = (raw_score > t4_flag_threshold).astype(int)
    print(f'Flagged by Test 4: {master["t4_discourse_flag"].sum():,} essays')
    print(f'  (discourse composite score > 1.5 SD above mean)')
else:
    master['t4_score'] = 0.5
    master['t4_discourse_flag'] = 0
    print('Warning: discourse features not found in master — Test 4 score set to 0.5.')

# ── Scatter: coherence vs. arg_density ───────────────────────────────────────
if 'coherence_score' in disc_feats and 'arg_density' in disc_feats:
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(master['arg_density'], master['coherence_score'],
                    c=master['t4_score'], cmap='YlOrRd', alpha=0.2, s=8)
    ax.scatter(syn_disc['arg_density'], syn_disc['coherence_score'],
               color='orange', alpha=0.4, s=15, marker='x', label='synthetic ref')
    plt.colorbar(sc, ax=ax, label='t4_score')
    ax.set_xlabel('arg_density')
    ax.set_ylabel('coherence_score')
    ax.set_title('Test 4: Discourse patterns (real essays vs. synthetic reference)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

pd.DataFrame([{
    'features_used': ', '.join(disc_feats),
    'n_flagged': int(master['t4_discourse_flag'].sum()),
    'mean_real_coherence': master['coherence_score'].mean() if 'coherence_score' in master else None,
    'mean_syn_coherence': syn_disc['coherence_score'].mean() if 'coherence_score' in syn_disc.columns else None,
}]).to_csv(OUTPUT_DIR / 'test_summaries/t4_discourse_summary.csv', index=False)

## Test 5: Combined Risk Score + Flagging

Combines the four test scores into a single `ai_risk_score` using a weighted linear combination.
Initial weights are based on expected signal strength from the literature.
**Revise weights after running Section 8 (stylometrics) and Section 6 (discourse) discriminability analyses.**

| Test | Initial weight | Rationale |
|---|---|---|
| T1 — Cosine similarity | 0.25 | Topic overlap can inflate scores |
| T2 — Perplexity/burstiness | 0.30 | Strongest literature-validated signal |
| T3 — Stylometric flatness | 0.30 | Strong signal, no new inference |
| T4 — Discourse patterns | 0.15 | Subtler signal, revise if AUC is strong |

In [ ]:
# ── Weights — revise after running discriminability sections ──────────────────
W1, W2, W3, W4 = 0.25, 0.30, 0.30, 0.15

FLAG_THRESHOLD = 0.7   # combined score threshold for flagging (tune based on distribution)

for col in ['t1_score', 't2_score', 't3_score', 't4_score']:
    if col not in master.columns:
        print(f'Warning: {col} missing — using 0.5 as placeholder')
        master[col] = 0.5

master['ai_risk_score'] = (
    W1 * master['t1_score'] +
    W2 * master['t2_score'] +
    W3 * master['t3_score'] +
    W4 * master['t4_score']
)
master['ai_risk_flag'] = (master['ai_risk_score'] > FLAG_THRESHOLD).astype(int)

print(f'Combined ai_risk_score statistics:')
print(master['ai_risk_score'].describe().round(3))
print(f'\nFlagged (score > {FLAG_THRESHOLD}): {master["ai_risk_flag"].sum():,} essays '
      f'({master["ai_risk_flag"].mean()*100:.1f}%)')

In [ ]:
# ── Validation: compute ai_risk_score on synthetic corpus ─────────────────────
from sklearn.metrics import roc_auc_score

# Redefine disc_zscore here using feat_means/feat_stds (covers all stylometric features)
# Cell 13's disc_zscore only covers discourse features — not sufficient here
def disc_zscore(col, feat):
    return (col - feat_means[feat]) / feat_stds[feat]

# Build synthetic feature df using the same thresholds
syn_stylo_val = stylo[stylo['corpus'].isin(['synthetic', 'freeform'])].copy()

# T3 score for synthetic
syn_cond_cv     = syn_stylo_val['cv_sent_len'] < t3_cv_threshold
syn_cond_mtld   = syn_stylo_val['mtld'] > t3_mtld_threshold
syn_cond_hollow = syn_stylo_val['hollow_phrase_density'] > t3_hollow_threshold
syn_stylo_val['t3_score_val'] = (
    (  disc_zscore(syn_stylo_val['hollow_phrase_density'], 'hollow_phrase_density')
     + disc_zscore(syn_stylo_val['intensifier_density'],   'intensifier_density')
     + disc_zscore(syn_stylo_val['rhetorical_q_rate'],     'rhetorical_q_rate')
     - disc_zscore(syn_stylo_val['cv_sent_len'],           'cv_sent_len')
    ) / 4
)

# T3-only AUC as a quick validation check
t3_only_val = pd.concat([
    master[['t3_score']].assign(label=0),
    syn_stylo_val[['t3_score_val']].rename(columns={'t3_score_val': 't3_score'}).assign(label=1)
])
t3_auc = roc_auc_score(t3_only_val['label'], t3_only_val['t3_score'])
print(f'Test 3 (stylometric flatness) AUC on real vs. synthetic: {t3_auc:.3f}')
print('Note: Full combined-score validation requires perplexity on synthetic (run Test 2 on ref pool first)')

In [ ]:
# ── Score distribution and score–reviewer-score correlation ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of combined risk score
axes[0].hist(master['ai_risk_score'], bins=60, color='#4C72B0', alpha=0.8)
axes[0].axvline(FLAG_THRESHOLD, color='red', linestyle='--', label=f'Flag threshold ({FLAG_THRESHOLD})')
axes[0].set_xlabel('ai_risk_score')
axes[0].set_ylabel('Number of essays')
axes[0].set_title('Combined AI risk score distribution')
axes[0].legend()

# Correlation with reviewer score (if available)
if 'score' in master.columns:
    axes[1].scatter(master['ai_risk_score'], master['score'],
                    alpha=0.1, s=5, color='#4C72B0')
    corr = master[['ai_risk_score', 'score']].corr().iloc[0, 1]
    axes[1].set_xlabel('ai_risk_score')
    axes[1].set_ylabel('Reviewer score')
    axes[1].set_title(f'AI risk score vs. reviewer score (r={corr:.3f})')
else:
    axes[1].text(0.5, 0.5, 'score column not available', ha='center', va='center')

plt.tight_layout()
plt.show()

# Breakdown by discourse cluster
if 'discourse_cluster' in master.columns:
    print('\nMean ai_risk_score by discourse cluster:')
    print(master.groupby('discourse_cluster')['ai_risk_score']
          .agg(['mean', 'count']).round(3))

# Breakdown by style cluster
if 'style_cluster' in master.columns:
    print('\nMean ai_risk_score by style cluster:')
    print(master.groupby('style_cluster')['ai_risk_score']
          .agg(['mean', 'count']).round(3))

In [ ]:
# ── Privacy-safe outputs ───────────────────────────────────────────────────────
score_cols = ['student_id', 'score', 'ai_risk_score', 'ai_risk_flag',
              't1_score', 't2_score', 't3_score', 't4_score',
              't2_perplexity_flag', 't3_flatness_flag', 't4_discourse_flag',
              'discourse_cluster', 'style_cluster']
score_cols = [c for c in score_cols if c in master.columns]

# Internal: includes student_id — gitignored
master[score_cols].to_csv(OUTPUT_DIR / 'detection_flags.csv', index=False)
print(f'Internal output saved: outputs/detection_flags.csv ({len(master):,} rows)')
print('  ← This file is gitignored. Do NOT share externally.')

# Shareable: aggregate statistics only, no student_id
summary_rows = []
for cluster_col in ['discourse_cluster', 'style_cluster']:
    if cluster_col in master.columns:
        grp = master.groupby(cluster_col).agg(
            n=('ai_risk_score', 'count'),
            mean_risk=('ai_risk_score', 'mean'),
            pct_flagged=('ai_risk_flag', 'mean'),
        ).reset_index()
        grp['groupby'] = cluster_col
        summary_rows.append(grp.rename(columns={cluster_col: 'cluster'}))

overall = pd.DataFrame([{
    'groupby': 'overall', 'cluster': 'all',
    'n': len(master),
    'mean_risk': master['ai_risk_score'].mean(),
    'pct_flagged': master['ai_risk_flag'].mean(),
}])
summary_rows.insert(0, overall)

detection_summary = pd.concat(summary_rows, ignore_index=True)
detection_summary.to_csv(OUTPUT_DIR / 'detection_summary.csv', index=False)
print(f'\nShareable summary saved: outputs/detection_summary.csv (no student IDs)')
print(detection_summary.round(3))

## Optional Section 5b: Trained Classifier

If the combined linear score AUC is below 0.85, fit a logistic regression or XGBoost
on the four test scores. Keep this optional — the linear combination is more auditable.

In [ ]:
# ── Test 5b: Logistic Regression trained on real vs. synthetic essays ─────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Rate-based features only — no raw counts that are confounded by essay length
LR_FEATURES = [f for f in [
    'mtld', 'hdd', 'avg_word_length', 'anaphora_proxy',
    'subordination_ratio', 'rhetorical_q_rate',
    'hollow_phrase_density', 'decorative_adj_density', 'cv_sent_len'
] if f in stylo.columns]

# Training set: real essays (label=0) + synthetic/freeform (label=1)
real_lr = stylo[stylo['corpus'] == 'real'][LR_FEATURES].copy().assign(label=0)
syn_lr  = stylo[stylo['corpus'].isin(['synthetic', 'freeform'])][LR_FEATURES].copy().assign(label=1)
train_df = pd.concat([real_lr, syn_lr], ignore_index=True).dropna(subset=LR_FEATURES)
X_lr = train_df[LR_FEATURES].values
y_lr = train_df['label'].values

pipe_lr = Pipeline([
    ('sc', StandardScaler()),
    ('lr', LogisticRegression(C=1, max_iter=1000, class_weight='balanced',
                               random_state=RANDOM_STATE))
])

cv_lr = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
auc_scores = cross_val_score(pipe_lr, X_lr, y_lr, cv=cv_lr, scoring='roc_auc')
print(f'Test 5b — Logistic Regression (real vs. synthetic/freeform)')
print(f'  Features ({len(LR_FEATURES)}): {LR_FEATURES}')
print(f'  5-fold CV AUC: {auc_scores.mean():.3f} +/- {auc_scores.std():.3f}')

# Fit on full training set for coefficients and essay scoring
pipe_lr.fit(X_lr, y_lr)
coefs = pipe_lr.named_steps['lr'].coef_[0]
coef_df = (pd.DataFrame({'feature': LR_FEATURES, 'coefficient': coefs})
             .sort_values('coefficient'))

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c' if c > 0 else '#3498db' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='none')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient  (positive = pushes toward synthetic)')
ax.set_title(f'Test 5b: Logistic regression coefficients\n5-fold CV AUC = {auc_scores.mean():.3f}')
plt.tight_layout()
plt.show()

# Score real essays: probability of being synthetic
real_feats = master[LR_FEATURES].fillna(master[LR_FEATURES].median())
master['t5b_lr_score'] = pipe_lr.predict_proba(real_feats.values)[:, 1]
print(f'\nt5b_lr_score statistics (real essays):')
print(master['t5b_lr_score'].describe().round(3))